In [1]:
from vllm import LLM, SamplingParams
import torch
import time

# Enable TF32 for better performance
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

MODEL_NAME = "google/gemma-2-9b-it"
#MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

model = LLM(model=MODEL_NAME, dtype=torch.bfloat16, trust_remote_code=True, quantization="bitsandbytes", load_format="bitsandbytes", max_model_len=8192)

INFO 02-11 13:30:36 __init__.py:190] Automatically detected platform cuda.
INFO 02-11 13:30:42 config.py:542] This model supports multiple tasks: {'score', 'generate', 'reward', 'classify', 'embed'}. Defaulting to 'generate'.
WARNING 02-11 13:30:42 config.py:621] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 02-11 13:30:42 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='google/gemma-2-9b-it', speculative_config=None, tokenizer='google/gemma-2-9b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.BITSANDBYTES, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=bitsandbytes, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_deco

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 02-11 13:30:50 model_runner.py:1115] Loading model weights took 5.7564 GB
INFO 02-11 13:30:57 worker.py:267] Memory profiling takes 6.49 seconds
INFO 02-11 13:30:57 worker.py:267] the current vLLM instance can use total_gpu_memory (24.00GiB) x gpu_memory_utilization (0.90) = 21.60GiB
INFO 02-11 13:30:57 worker.py:267] model weights take 5.76GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 2.38GiB; the rest of the memory reserved for KV Cache is 13.40GiB.
INFO 02-11 13:30:57 executor_base.py:110] # CUDA blocks: 2614, # CPU blocks: 780
INFO 02-11 13:30:57 executor_base.py:115] Maximum concurrency for 8192 tokens per request: 5.11x
INFO 02-11 13:30:57 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utili

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:21<00:00,  1.63it/s]

INFO 02-11 13:31:19 model_runner.py:1562] Graph capturing finished in 22 secs, took 0.78 GiB
INFO 02-11 13:31:19 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 28.72 seconds


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Prepare the chat message
messages = [
    {"role": "user", "content": "Write me a poem about Machine Learning."},
]

# Convert messages to prompt string
# Note: Adjust template based on model's expected format
prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Set sampling parameters
sampling_params = SamplingParams(
    max_tokens=256,
    temperature=1.0,
    min_p=0.1
)

# Generate with timing
start_time = time.time()
outputs = model.generate(prompt, sampling_params)
end_time = time.time()

# Print the generated text
generated_text = outputs[0].outputs[0].text
print("\nGenerated Text:")
print(generated_text)

# Calculate and print stats
generation_time = end_time - start_time
num_new_tokens = len(outputs[0].outputs[0].token_ids)
tokens_per_second = num_new_tokens / generation_time

print(f"\nGeneration Stats:")
print(f"Total tokens generated: {num_new_tokens}")
print(f"Generation time: {generation_time:.2f} seconds")
print(f"Speed: {tokens_per_second:.2f} tokens/second")

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.66s/it, est. speed input: 4.10 toks/s, output: 60.38 toks/s]


Generated Text:


In silicon valleys, where circuits gleam,
A new intelligence takes form, a waking dream.
Machine Learning, a dance of data's might,
Unveiling patterns, hidden from human sight.

Algorithms churn, with tireless grace,
Through terabytes of information's embrace.
Each input a whisper, a clue to impart,
Training the model, a work of digital art.

From pixels to prose, from sound to speech,
It learns to discern, to reason, to teach.
Classifying images, predicting the trend,
A symphony of logic, that knows no end.

But caution whispers, a voice soft and low,
Of biases ingrained, and futures we don't know.
For data reflects our world, its flaws and its fears,
And algorithms echo, through digital tears.

So let us guide its hand, with wisdom and care,
Ensure its purpose is noble, beyond compare.
Machine Learning, a tool of immense might,
May it illuminate the future, with wisdom and light.





Generation Stats:
Total tokens generated: 221
Generation time: 3.66 seconds
Speed